In [1]:
import torch
import torchaudio
import numpy as np
from pathlib import Path
from tqdm import tqdm

from speechbrain.inference import EncoderClassifier


/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [ ]:
import urllib.request
from pathlib import Path

# Where your test data lives
DATA_ROOT = Path("../data/voxceleb1/test")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

veri_url = "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt"
veri_path = DATA_ROOT / "veri_test2.txt"

if not veri_path.exists():
    print("Downloading veri_test2.txt ...")
    urllib.request.urlretrieve(veri_url, veri_path)
    print("Downloaded to:", veri_path)
else:
    print("veri_test2.txt already exists at:", veri_path)

In [2]:
# Load ECAPA-TDNN pretrained on VoxCeleb with AAM-Softmax
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}  # use "cuda" if available
)


hyperparams.yaml: 1.92kB [00:00, 162kB/s]
/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
embedding_model.ckpt: 100%|██████████| 83.3M/83.3M [00:11<00:00, 7.52MB/s]
m

In [3]:
DATA_ROOT = Path("../data/voxceleb1/test/wav")
OUT_DIR = Path("../outputs/ecapa_xvectors")
OUT_DIR.mkdir(parents=True, exist_ok=True)

wav_files = sorted(DATA_ROOT.rglob("*.wav"))

print("Total wav files:", len(wav_files))
print("Example:", wav_files[0])


Total wav files: 4874
Example: ../data/voxceleb1/test/wav/id10270/5r0dWxy17C8/00001.wav


In [4]:
X = []
utterances = []
speakers = []

for wav_path in tqdm(wav_files):
    # Load audio
    signal, sr = torchaudio.load(wav_path)
    
    # Convert to mono
    if signal.shape[0] > 1:
        signal = signal.mean(dim=0, keepdim=True)

    # Extract ECAPA embedding (192-dim)
    with torch.no_grad():
        emb = classifier.encode_batch(signal).squeeze().cpu().numpy()

    X.append(emb)

    # Relative path used in protocol
    rel = wav_path.relative_to(DATA_ROOT)
    utterances.append(str(rel))
    speakers.append(rel.parts[0])  # id10270

X = np.vstack(X)

print("X-vectors shape:", X.shape)


  0%|          | 0/4874 [00:00<?, ?it/s]/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
100%|██████████| 4874/4874 [05:02<00:00, 16.09it/s]

X-vectors shape: (4874, 192)


In [6]:
np.save(OUT_DIR / "xvectors.npy", X)
np.save(OUT_DIR / "utterances.npy", np.array(utterances))
np.save(OUT_DIR / "speakers.npy", np.array(speakers))

print("Saved ECAPA x-vectors")


Saved ECAPA x-vectors
